<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_10_Real_Provenance_TRACE_Athena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, json, re, csv, io, zipfile, hashlib, platform, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import pandas as pd

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

BASE=Path('/content') if Path('/content').exists() else Path.cwd()
PROJECT=Path(os.environ.get('ICHI_PROJECT_DIR',BASE/'drive/MyDrive/NeuroFHIR_ICHI2027'))
OUT=PROJECT/'phase10_outputs'
OUT.mkdir(parents=True,exist_ok=True)
PRIVATE=OUT/'_private_recovery'
PRIVATE.mkdir(parents=True,exist_ok=True)
REPORT=OUT/'ICHI2027_Phase10_SHAREABLE.json'
SOURCE_NAME='TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
PHASE09_NAME='ICHI2027_Phase09_SHAREABLE.json'

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def now(): return datetime.now(timezone.utc).isoformat()
def file_globs(d,patterns):
    if not d.exists(): return []
    return sorted({p.resolve() for pat in patterns for p in d.glob(pat) if p.is_file()},key=str)

def header_csv(path):
    with open(path,'r',encoding='utf-8-sig',errors='replace',newline='') as f:
        line=f.readline()
    sep='\t' if line.count('\t')>line.count(',') else ','
    return [c.strip() for c in next(csv.reader([line],delimiter=sep))],sep

Mounted at /content/drive


In [2]:
prior_candidates=[
    Path(os.environ['ICHI_PHASE09_PATH']) if 'ICHI_PHASE09_PATH' in os.environ else None,
    PROJECT/'phase09_outputs'/PHASE09_NAME,
    BASE/PHASE09_NAME,
    BASE/'inputs'/PHASE09_NAME,
]
prior_file=next((p for p in prior_candidates if p is not None and p.is_file()),None)
if prior_file is None:
    from google.colab import files
    uploaded=files.upload()
    if PHASE09_NAME not in uploaded:
        raise FileNotFoundError(PHASE09_NAME)
    prior_file=PRIVATE/PHASE09_NAME
    prior_file.write_bytes(uploaded[PHASE09_NAME])
prior=json.loads(prior_file.read_text(encoding='utf-8'))
assert prior['phase']=='09_actual_fhir_to_private_omop_candidate_etl'
assert prior['gates']['actual_fhir_to_private_candidate_etl_completed']
assert prior['gates']['official_fhir_r4_validation_passed']
PHASE09_SHA=sha256_file(prior_file)
assert prior['private_candidate_etl']['fhir_numeric_roundtrip_decimal_mismatches']==0
assert prior['private_candidate_etl']['foreign_key_failures']==0

In [3]:
source_candidates=[BASE/SOURCE_NAME,BASE/'inputs'/SOURCE_NAME,
                   PROJECT/'inputs'/SOURCE_NAME,PROJECT/'source_data'/SOURCE_NAME]
source_path=next((p for p in source_candidates if p.is_file()),None)
source_profile={'present':False,'schema_equal_to_verified_columns':None,
                'rows_equal_to_phase09':None,'numeric_feature_event_count_reconciled':None,
                'unit_column_present':False,'verified_dicom_columns_present':False,
                'source_hash_saved_privately':False}
SOURCE_HASH=None
if source_path:
    source=pd.read_csv(source_path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    source.columns=[c.strip() for c in source.columns]
    expected=['RID','PTID']+[f'{tp}_{field}' for tp in ('BL','M12') for field in
         ('VISCODE','EXAMDATE','IMAGEUID','LONIUID','VERSION','OVERALLQC',
          'LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV')]
    missing=sorted(set(expected)-set(source.columns))
    rowmatch=len(source)==prior['source']['rows']
    event_count=sum(source[f'{tp}_{f}'].astype(str).str.strip().ne('').sum() for tp in ('BL','M12')
                    for f in ('LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV')
                    if f'{tp}_{f}' in source)
    SOURCE_HASH=sha256_file(source_path)
    source_profile={
        'present':True,'schema_equal_to_verified_columns':not missing and len(source.columns)==prior['source']['schema_fields'],
        'missing_required_columns':missing,'rows_equal_to_phase09':rowmatch,
        'numeric_feature_event_count_reconciled':int(event_count)==prior['source']['feature_events'],
        'unit_column_present':any('UNIT' in c.upper() for c in source.columns),
        'verified_dicom_columns_present':{'BL_STUDYINSTANCEUID','BL_SERIESINSTANCEUID',
                                           'M12_STUDYINSTANCEUID','M12_SERIESINSTANCEUID'}.issubset(source.columns),
        'source_hash_saved_privately':True,
    }
    (PRIVATE/'source_reproducibility_manifest.json').write_text(json.dumps({
        'source_path_private':str(source_path),'source_sha256_private':SOURCE_HASH,
        'phase09_sha256':PHASE09_SHA,'verified_source_headers_sha256':
        hashlib.sha256('\n'.join(source.columns).encode()).hexdigest(),
        'run_utc':now()
    },indent=2))
else:
    source=None

In [4]:
metadata_dirs=[PROJECT/'source_metadata',PROJECT/'source_docs',PROJECT/'trace_outputs',
               PROJECT/'_private_reproducibility',PROJECT/'vocabulary',PROJECT/'inputs'/'metadata']
review_exts={'.csv','.tsv','.json','.parquet','.xlsx','.pdf','.md','.txt','.zip'}
metadata_files=[]
for folder in metadata_dirs:
    if folder.is_dir():
        for p in folder.rglob('*'):
            if p.is_file() and p.suffix.lower() in review_exts and p.stat().st_size<300_000_000:
                metadata_files.append(p)
metadata_files=sorted(set(metadata_files),key=str)[:250]
metadata_inventory={'candidate_files':len(metadata_files),'by_extension':{},
                    'trace_candidate_files':0,'source_dictionary_candidate_files':0,
                    'dicom_metadata_candidate_files':0}
for p in metadata_files:
    ext=p.suffix.lower()
    metadata_inventory['by_extension'][ext]=metadata_inventory['by_extension'].get(ext,0)+1
    n=p.name.lower()
    if any(t in n for t in ('trace','reliability','calibration')):
        metadata_inventory['trace_candidate_files']+=1
    if any(t in n for t in ('dictionary','ucsf','units','metadata','freesurfer','provenance')):
        metadata_inventory['source_dictionary_candidate_files']+=1
    if any(t in n for t in ('dicom','studyuid','seriesuid','crosswalk')):
        metadata_inventory['dicom_metadata_candidate_files']+=1

In [5]:
trace_profiles=[]
for p in metadata_files:
    if p.suffix.lower() not in {'.csv','.tsv','.parquet'}: continue
    try:
        if p.suffix.lower()=='.parquet':
            import pyarrow.parquet as pq
            names=pq.ParquetFile(p).schema.names
            rows=pq.ParquetFile(p).metadata.num_rows
        else:
            names,sep=header_csv(p)
            rows=None
    except Exception:
        continue
    lower={c.lower().strip():c for c in names}
    states=[c for c in names if re.search(r'(trace|reliability).*(state|status)|(state|status).*(trace|reliability)',c,re.I)]
    reasons=[c for c in names if re.search(r'(trace|reliability).*(reason|flag)|reason.*code',c,re.I)]
    references=[c for c in names if re.search(r'(trace|calibration).*(reference|ref|envelope|q9[05])',c,re.I)]
    rules=[c for c in names if re.search(r'(trace|reliability|rule).*(version|revision)',c,re.I)]
    ids=[c for c in names if c.lower().strip() in {'rid','ptid','person_id','participant_id'}]
    if not (states or references or rules): continue
    trace_profiles.append({
        'file_index':len(trace_profiles), 'private_file_sha256':sha256_file(p),
        'column_names':list(names),'state_columns':states,'reason_columns':reasons,
        'reference_columns':references,'rule_version_columns':rules,'join_key_columns':ids,'rows_if_parquet':rows,
        'complete_state_reference_rule_candidate':bool(states and references and rules and ids)
    })

# Column evidence alone does not establish that this file is an original, calibrated TRACE output.
trace_candidates_complete=sum(x['complete_state_reference_rule_candidate'] for x in trace_profiles)
(PRIVATE/'trace_candidate_files_private.json').write_text(json.dumps({
    'paths_private':{str(i):str(p) for i,p in enumerate(metadata_files)},
    'profiles':trace_profiles
},indent=2))

288

In [6]:
# Public repository inspection only; no user CSV is uploaded to GitHub.
REPO_URL='https://github.com/SANGHATI23/neurofhir-qc.git'
REPO_COMMIT='88c1a6b07420b62719ef9efcec7494a86446fa39'
repo_summary={'status':'NOT_CHECKED','pinned_commit':REPO_COMMIT,
              'actual_commit':None,'potential_trace_file_count':0,
              'evidence_for_calibrated_original_trace_output':False}
if os.environ.get('ICHI_SKIP_REMOTE')=='1':
    repo_summary['status']='SKIPPED_OFFLINE_TEST'
else:
    dest=PRIVATE/'neurofhir_qc_pinned'
    if (dest/'.git').exists():
        proc=subprocess.run(['git','rev-parse','--is-inside-work-tree'],cwd=dest,capture_output=True,text=True)
    else:
        if dest.exists(): shutil.rmtree(dest)
        proc=subprocess.run(['git','clone','-q',REPO_URL,str(dest)],capture_output=True,text=True)
    if proc.returncode==0:
        checkout=subprocess.run(['git','checkout','-q',REPO_COMMIT],cwd=dest,capture_output=True,text=True)
        if checkout.returncode==0:
            rev=subprocess.run(['git','rev-parse','HEAD'],cwd=dest,capture_output=True,text=True)
            repo_summary['actual_commit']=rev.stdout.strip()
            repo_summary['status']='PINNED' if rev.stdout.strip()==REPO_COMMIT else 'HASH_MISMATCH'
            plausible=[p for p in dest.rglob('*') if p.is_file() and any(
                x in str(p.relative_to(dest)).lower() for x in ('trace','calibration','reliability','adni'))]
            repo_summary['potential_trace_file_count']=len(plausible)
        else:
            repo_summary['status']='CHECKOUT_FAILED'
    else:
        repo_summary['status']='CLONE_FAILED'

In [7]:
# Public ADNI documentation is reference context, not evidence that a particular local export has a given unit.
public_docs={'url':'https://adni.loni.usc.edu/quick-start-guide-asset/MRI_tables.html',
             'status':'NOT_RETRIEVED','document_sha256':None,
             'mentions_ucsffsl51_in_visible_text':False,
             'mentions_freesurfer_version_5_1_in_visible_text':False,
             'explicit_export_specific_unit_verified':False}
if os.environ.get('ICHI_SKIP_REMOTE')=='1':
    public_docs['status']='SKIPPED_OFFLINE_TEST'
else:
    try:
        import requests
        from bs4 import BeautifulSoup
        response=requests.get(public_docs['url'],timeout=45,headers={'User-Agent':'Mozilla/5.0'})
        response.raise_for_status()
        public_docs['document_sha256']=hashlib.sha256(response.content).hexdigest()
        soup=BeautifulSoup(response.content,'html.parser')
        for node in soup(['script','style','noscript','svg','img']): node.decompose()
        visible=' '.join(soup.stripped_strings)
        public_docs['status']='RETRIEVED_AND_VISIBLE_TEXT_PARSED'
        public_docs['mentions_ucsffsl51_in_visible_text']='UCSFFSL51' in visible
        public_docs['mentions_freesurfer_version_5_1_in_visible_text']=bool(
            re.search(r'freesurfer.{0,30}(5[.]1|version\s*5[.]1)',visible,re.I))
        (PRIVATE/'public_adni_visible_text_sha256.json').write_text(json.dumps({
            'url':public_docs['url'],'html_sha256':public_docs['document_sha256'],
            'visible_text_sha256':hashlib.sha256(visible.encode()).hexdigest(),
            'checked_utc':now()
        },indent=2))
    except Exception as ex:
        public_docs['status']='DOWNLOAD_OR_PARSE_FAILED'
        public_docs['error_type']=type(ex).__name__

In [8]:
# The template is deliberately empty; populate outside the notebook only with documented source evidence.
CONTRACT=PROJECT/'source_metadata'/'verified_export_contract.json'
TEMPLATE=OUT/'SOURCE_PROVENANCE_CONTRACT_TEMPLATE.json'
empty_contract={
    'exact_original_adni_table_name':'',
    'original_download_filename':'',
    'local_processed_csv_sha256':'',
    'source_dictionary_relative_path':'',
    'source_dictionary_sha256':'',
    'dictionary_variable_or_section_for_volume_unit':'',
    'documented_volume_unit':'',
    'scan_uid_crosswalk_relative_path':'',
    'dicom_study_uid_field':'',
    'dicom_series_uid_field':'',
    'processing_software_document_relative_path':'',
    'processing_software_name':'',
    'processing_software_version_field':'',
    'original_trace_output_relative_path':'',
    'original_trace_state_field':'',
    'original_trace_rule_version_field':'',
    'original_trace_reference_field':'',
    'baseline_mci_source_relative_path':'',
    'baseline_mci_field':'',
}
if not TEMPLATE.exists(): TEMPLATE.write_text(json.dumps(empty_contract,indent=2))
provenance_review={'submitted_contract_found':CONTRACT.is_file(),
                   'source_csv_digest_matched':False,'source_dictionary_digest_matched':False,
                   'unit_claim_has_document_candidate':False,
                   'dicom_crosswalk_file_present':False,'trace_file_present':False,
                   'eligible_for_human_provenance_review':False,
                   'any_unit_or_uid_approved_by_this_notebook':False}
if CONTRACT.is_file():
    candidate=json.loads(CONTRACT.read_text())
    provenance_review['source_csv_digest_matched']=bool(SOURCE_HASH and
         candidate.get('local_processed_csv_sha256')==SOURCE_HASH)
    # Reject absolute paths, symlinks outside the controlled Drive source_metadata folder and traversal.
    def local_metadata_path(value):
        if not value: return None
        d=(PROJECT/'source_metadata').resolve()
        p=(d/value).resolve()
        return p if p.is_relative_to(d) and p.is_file() else None
    doc=local_metadata_path(candidate.get('source_dictionary_relative_path'))
    provenance_review['source_dictionary_digest_matched']=bool(doc and
         candidate.get('source_dictionary_sha256')==sha256_file(doc))
    provenance_review['unit_claim_has_document_candidate']=bool(
        provenance_review['source_dictionary_digest_matched'] and
        candidate.get('dictionary_variable_or_section_for_volume_unit') and
        candidate.get('documented_volume_unit'))
    provenance_review['dicom_crosswalk_file_present']=bool(
        local_metadata_path(candidate.get('scan_uid_crosswalk_relative_path')))
    provenance_review['trace_file_present']=bool(
        local_metadata_path(candidate.get('original_trace_output_relative_path')))
    provenance_review['eligible_for_human_provenance_review']=bool(
        provenance_review['source_csv_digest_matched'] and
        provenance_review['unit_claim_has_document_candidate'])

In [9]:
# Genuine Athena archives only. No concept is automatically approved by matching its name.
VOC_DIRS=[PROJECT/'vocabulary',BASE/'vocabulary']
vocab_packages=[]
for d in VOC_DIRS:
    vocab_packages+=file_globs(d,['*.zip','CONCEPT.csv','VOCABULARY.csv'])
zip_packages=[p for p in vocab_packages if p.suffix.lower()=='.zip']
zip_packages=sorted(zip_packages,key=lambda p:p.stat().st_mtime,reverse=True)
VOC={'snapshot_found':False,'package_sha256':None,'status':'NO_ATHENA_SNAPSHOT',
     'full_package_files_present':False,
     'concept_rows_scanned':0,'vocabulary_versions_found':0,
     'standard_valid_candidates_by_family':{},'manual_review_required':True}
concept_stream=None
vocab_stream=None
package=None
for z in zip_packages:
    try:
        trial=zipfile.ZipFile(z)
        ns={Path(n).name.upper():n for n in trial.namelist() if not n.endswith('/')}
        required_tables={'CONCEPT.CSV','VOCABULARY.CSV','CONCEPT_ANCESTOR.CSV',
                         'CONCEPT_SYNONYM.CSV','CONCEPT_CLASS.CSV','DOMAIN.CSV',
                         'DRUG_STRENGTH.CSV','CONCEPT_RELATIONSHIP.CSV','RELATIONSHIP.CSV'}
        if 'CONCEPT.CSV' in ns and 'VOCABULARY.CSV' in ns:
            package=trial
            concept_stream=trial.open(ns['CONCEPT.CSV'])
            vocab_stream=trial.open(ns['VOCABULARY.CSV'])
            VOC['package_sha256']=sha256_file(z)
            VOC['snapshot_found']=True
            VOC['full_package_files_present']=required_tables.issubset(ns)
            break
        trial.close()
    except (zipfile.BadZipFile,OSError):
        continue
if not VOC['snapshot_found']:
    for d in VOC_DIRS:
        cp=d/'CONCEPT.csv'; vp=d/'VOCABULARY.csv'
        if cp.is_file() and vp.is_file():
            concept_stream=open(cp,'rb'); vocab_stream=open(vp,'rb')
            VOC['snapshot_found']=True
            VOC['package_sha256']=hashlib.sha256((sha256_file(cp)+sha256_file(vp)).encode()).hexdigest()
            break

candidate_records=[]
if VOC['snapshot_found']:
    def csv_sep(f):
        pos=f.tell(); header=f.readline().decode('utf-8-sig','replace'); f.seek(pos)
        return '\t' if header.count('\t')>=header.count(',') else ','
    sep=csv_sep(vocab_stream)
    version_records=[]
    for chunk in pd.read_csv(vocab_stream,sep=sep,dtype=str,keep_default_na=False,
                             chunksize=20000,encoding='utf-8-sig'):
        if {'vocabulary_id','vocabulary_version'}.issubset(chunk.columns):
            version_records.extend(chunk[['vocabulary_id','vocabulary_version']].to_dict('records'))
    VOC['vocabulary_versions_found']=len(version_records)
    (PRIVATE/'athena_versions_private.json').write_text(json.dumps(version_records,indent=2))
    vocab_stream.close()
    family_patterns={
        'MRI_modality':r'magnetic resonance imaging|magnetic resonance',
        'hippocampus':r'hippocamp',
        'entorhinal':r'entorhinal',
        'middle_temporal':r'middle temporal',
        'ventricles':r'cerebral ventricle|lateral ventricle',
        'intracranial_volume':r'intracranial volume',
        'volume_unit':r'cubic millimeter|millimeter cubed|mm3|mm\^3',
    }
    counts={k:0 for k in family_patterns}
    sep=csv_sep(concept_stream)
    for chunk in pd.read_csv(concept_stream,sep=sep,dtype=str,keep_default_na=False,
                             chunksize=150000,encoding='utf-8-sig'):
        required={'concept_id','concept_name','domain_id','vocabulary_id',
                  'standard_concept','invalid_reason'}
        if not required.issubset(chunk.columns):
            VOC['status']='CONCEPT_REQUIRED_COLUMNS_MISSING'
            break
        VOC['concept_rows_scanned']+=len(chunk)
        valid=chunk[(chunk['standard_concept']=='S') & (chunk['invalid_reason']=='')]
        for family,pat in family_patterns.items():
            matching=valid[valid['concept_name'].str.contains(pat,case=False,regex=True,na=False)]
            counts[family]+=len(matching)
            for record in matching.head(max(0,30-sum(x['family']==family for x in candidate_records))).to_dict('records'):
                candidate_records.append({
                    'family':family,'concept_id':record['concept_id'],
                    'concept_name':record['concept_name'],'domain_id':record['domain_id'],
                    'vocabulary_id':record['vocabulary_id'],
                    'approval_status':'UNREVIEWED',
                    'mapped_source_field':'','evidence_document':'',
                })
    concept_stream.close()
    if package is not None: package.close()
    VOC['standard_valid_candidates_by_family']=counts
    if VOC['status']!='CONCEPT_REQUIRED_COLUMNS_MISSING':
        VOC['status']=('FULL_ATHENA_PACKAGE_SCANNED_UNAPPROVED_CANDIDATES' if
                        VOC['full_package_files_present'] else
                        'PARTIAL_ATHENA_TABLES_SCANNED_UNAPPROVED_CANDIDATES')
    if candidate_records:
        pd.DataFrame(candidate_records).to_csv(PRIVATE/'athena_mapping_candidates_UNAPPROVED_PRIVATE.csv',index=False)

In [10]:
# All output gates are evidence-aware and intentionally remain closed until human approval.
REPORT_DATA={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'10_provenance_trace_and_athena_evidence_recovery',
 'run_utc':now(),
 'result_status':'PHASE09_REAL_ETL_CONFIRMED_PROVENANCE_AND_TERMINOLOGY_RECOVERY_ONLY',
 'privacy':'Only aggregates and public repository metadata shared; private source hashes, paths and candidates stay in Drive.',
 'phase09_json_sha256':PHASE09_SHA,
 'verified_phase09_inherited':{
     'fhir_r4_validation':prior['fhir_r4_validation']['validation'],
     'real_candidate_measurement_events':prior['private_candidate_etl']['measurement_candidate_rows'],
     'numeric_mismatches':prior['private_candidate_etl']['fhir_numeric_roundtrip_decimal_mismatches'],
     'official_micdm_load':prior['gates']['official_micdm_load_completed'],
 },
 'optional_source_reverification':source_profile,
 'private_document_inventory':metadata_inventory,
 'original_trace_sidecar_candidates':{
     'potential_profile_count':len(trace_profiles),
     'profiles_with_state_reference_rule_and_join_key':trace_candidates_complete,
     'original_trace_output_provenance_verified':False,
 },
 'pinned_neurofhir_qc_repo_inspection':repo_summary,
 'official_public_adni_reference':public_docs,
 'source_export_evidence_contract':provenance_review,
 'athena_vocabulary':VOC,
 'gates':{
     'phase09_actual_fhir_candidate_etl_confirmed':True,
     'exact_local_export_units_authoritatively_verified':False,
     'original_trace_state_and_calibration_verified':False,
     'dicom_study_and_series_uids_verified':False,
     'full_athena_snapshot_scanned':VOC['status']=='FULL_ATHENA_PACKAGE_SCANNED_UNAPPROVED_CANDIDATES',
     'omop_concept_mappings_approved':False,
     'baseline_MCI_eligibility_approved':False,
     'official_micdm_load_completed':False,
     'real_four_arm_comparison_completed':False,
 },
 'four_arm_comparator_scores':None,
 'official_micdm_load_results':None,
 'next_stage_gate':None,
}
if not VOC['snapshot_found']:
    REPORT_DATA['next_stage_gate']='OBTAIN_ATHENA_FULL_VOCABULARY_PACKAGE_AND_SOURCE_EXPORT_DICTIONARY_PLUS_TRACE_SIDECAR'
elif not provenance_review['eligible_for_human_provenance_review']:
    REPORT_DATA['next_stage_gate']='REVIEW_AUTHORITATIVE_EXACT_SOURCE_DICTIONARY_AND_TRACE_PROVENANCE'
else:
    REPORT_DATA['next_stage_gate']='HUMAN_REVIEW_SOURCE_PROVENANCE_AND_ATHENA_CONCEPT_CANDIDATES_BEFORE_ANY_OFFICIAL_LOAD'
REPORT.write_text(json.dumps(REPORT_DATA,indent=2),encoding='utf-8')
print(json.dumps({'phase':REPORT_DATA['phase'],'candidate_rows':REPORT_DATA['verified_phase09_inherited']['real_candidate_measurement_events'],
    'source_reverified':source_profile['present'],'athena_status':VOC['status'],
    'trace_sidecar_candidates':trace_candidates_complete,
    'next_stage_gate':REPORT_DATA['next_stage_gate'],
    'saved_report':str(REPORT)},indent=2))

{
  "phase": "10_provenance_trace_and_athena_evidence_recovery",
  "candidate_rows": 4128,
  "source_reverified": false,
  "athena_status": "NO_ATHENA_SNAPSHOT",
  "trace_sidecar_candidates": 0,
  "next_stage_gate": "OBTAIN_ATHENA_FULL_VOCABULARY_PACKAGE_AND_SOURCE_EXPORT_DICTIONARY_PLUS_TRACE_SIDECAR",
  "saved_report": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase10_outputs/ICHI2027_Phase10_SHAREABLE.json"
}


In [11]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(REPORT))
else:
    print(str(REPORT))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>